# 22 — 29042675 DOCX Label Recovery

## Objetivo

Recuperar de forma reproducible las evaluaciones H1, H4, H5 y H6 del dataset `29042675` después de que el notebook 21 demostrara que:

- `H* Compare` pertenece al análisis de acuerdo/IRR y **no debe usarse como etiqueta de violación**;
- existen 43 documentos de evaluaciones LLM;
- una parte de los documentos de la segunda ronda no usa la estructura DOCX esperada `word/document.xml`;
- la extracción previa encontró secciones heurísticas, pero no logró recuperar severidades porque tomaba la primera aparición del encabezado, que puede pertenecer al prompt.

Este notebook corrige ambos problemas:

1. audita y recupera texto de distintos tipos de paquetes ZIP/XML;
2. selecciona la **última aparición** de cada heurística para evitar confundir el prompt con la respuesta;
3. conserva la procedencia de cada unión `documento ↔ prototipo`;
4. genera únicamente **candidatos LLM_DIRECT**, nunca X del modelo.


## 1. Configuración

In [1]:
from pathlib import Path
from urllib.parse import urlparse, unquote
from collections import defaultdict
from difflib import SequenceMatcher
import pandas as pd
import numpy as np
import re
import zipfile
import xml.etree.ElementTree as ET
import html

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

PUBLIC_RAW = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "raw"
)

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "interim"
)

VT_DIR = (
    PUBLIC_RAW
    / "03_29042675"
)

LLM_DIR = (
    VT_DIR
    / "LLM_Evaluations"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("VT_DIR:", VT_DIR)


VT_DIR: /Users/uriel/Documents/machine_learning/data/public_v2/raw/03_29042675


## 2. Cargar metadata

In [2]:
metadata = pd.read_csv(
    VT_DIR
    / "Prototype_Metadata.csv"
)

print(
    "Metadata:",
    metadata.shape
)

display(
    metadata
)


Metadata: (20, 4)


,Prototype,Prototype Type,Domain,Scource URL
0,Food Order,High Fidelity,Delivery,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App--Community-?node-id=0-1&t=h3GPGrBwJYFFbxI3-0
1,Gaira (Community app),High Fidelity,Community,https://www.figma.com/design/1WJTuXkM0OfRiWjZkD61n2/Gaia-Desktop-Design-(Community)?node-id=0-1&p=f&t=kawo30RBorPkO1d2-0?
2,E-Tutor,High Fidelity,Education,https://www.figma.com/community/file/1271304360859402063
3,Tracking App,High Fidelity,Mobile app,https://www.behance.net/gallery/198020739/Time-Tracker-mobile-application?tracking_source=project_owner_other_projects
4,Travel,High Fidelity,Travel,https://www.figma.com/community/file/1379689864793179441
5,Running,High Fidelity,Healthcare,https://www.behance.net/gallery/158363369/UI-Projects/modules/893495869
6,Bakery,High Fidelity,Food Delivey,https://www.figma.com/community/file/988854619809579450
7,Hospital 1,High Fidelity,Healthcare,https://www.figma.com/community/file/1250314760992299521
8,Hospital 2,High Fidelity,Healthcare,https://www.figma.com/community/file/12725823068122925
9,Housing,High Fidelity,Real Estate,https://www.figma.com/design/QLcZcyPAov1fsHDEfnX6T4/Real-Estate-Agency---Landing-Page-Design-(Community)?node-id=0-1&p=f&t=IBZxitN7XpPIezha-0


## 3. Inventario de los 43 documentos

In [3]:
docx_files = sorted(
    [
        path
        for path
        in LLM_DIR.rglob("*.docx")
        if path.is_file()
    ]
)

print(
    "DOCX encontrados:",
    len(docx_files)
)

assert len(docx_files) == 43


DOCX encontrados: 43


## 4. Auditar la estructura ZIP/XML

No se presupone que todos los archivos con extensión `.docx` tengan `word/document.xml`.


In [4]:
def inspect_zip_package(path):

    result = {
        "file":
            str(
                path.relative_to(
                    VT_DIR
                )
            ),

        "is_zip":
            False,

        "member_count":
            0,

        "has_word_document":
            False,

        "package_type":
            None,

        "candidate_text_members":
            None,

        "error":
            None
    }

    try:

        with zipfile.ZipFile(path) as z:

            result[
                "is_zip"
            ] = True

            names = z.namelist()

            result[
                "member_count"
            ] = len(names)

            lower_names = {
                name.lower():
                    name
                for name in names
            }

            result[
                "has_word_document"
            ] = (
                "word/document.xml"
                in lower_names
            )

            if any(
                name.lower().startswith(
                    "word/"
                )
                for name in names
            ):
                package_type = "WORD_PACKAGE"

            elif any(
                name.lower().startswith(
                    "ppt/"
                )
                for name in names
            ):
                package_type = "POWERPOINT_PACKAGE"

            elif any(
                name.lower().startswith(
                    "xl/"
                )
                for name in names
            ):
                package_type = "EXCEL_PACKAGE"

            elif any(
                Path(name).name.lower()
                ==
                "content.xml"
                for name in names
            ):
                package_type = "OPEN_DOCUMENT_PACKAGE"

            else:
                package_type = "UNKNOWN_ZIP_PACKAGE"

            result[
                "package_type"
            ] = package_type

            candidates = [
                name
                for name
                in names
                if (
                    name.lower().endswith(
                        ".xml"
                    )
                    and
                    (
                        "document"
                        in name.lower()
                        or
                        "slide"
                        in name.lower()
                        or
                        Path(name).name.lower()
                        ==
                        "content.xml"
                    )
                )
            ]

            result[
                "candidate_text_members"
            ] = " | ".join(
                candidates[
                    :30
                ]
            )

    except Exception as e:

        result[
            "error"
        ] = str(e)

    return result


In [5]:
package_audit = pd.DataFrame(
    [
        inspect_zip_package(
            path
        )
        for path
        in docx_files
    ]
)

print(
    package_audit[
        "package_type"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nword/document.xml:"
)

print(
    package_audit[
        "has_word_document"
    ].value_counts(
        dropna=False
    )
)

display(
    package_audit[
        [
            "file",
            "package_type",
            "has_word_document",
            "member_count",
            "candidate_text_members",
            "error"
        ]
    ]
)


package_type
WORD_PACKAGE    43
Name: count, dtype: int64

word/document.xml:
has_word_document
True     24
False    19
Name: count, dtype: int64


,file,package_type,has_word_document,member_count,candidate_text_members,error
0,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,WORD_PACKAGE,True,15,word/document.xml,None
1,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,WORD_PACKAGE,True,15,word/document.xml,None
2,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Food delivery.docx,WORD_PACKAGE,True,15,word/document.xml,None
3,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Gaia.docx,WORD_PACKAGE,True,15,word/document.xml,None
4,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp.docx,WORD_PACKAGE,True,15,word/document.xml,None
5,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp2.docx,WORD_PACKAGE,True,15,word/document.xml,None
6,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Housing.docx,WORD_PACKAGE,True,15,word/document.xml,None
7,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Notification.docx,WORD_PACKAGE,True,15,word/document.xml,None
8,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Running.docx,WORD_PACKAGE,True,15,word/document.xml,None
9,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Travel.docx,WORD_PACKAGE,True,15,word/document.xml,None


## 5. Extractor genérico XML

Extrae únicamente nodos de texto de miembros XML plausibles. El archivo y el miembro utilizado quedan registrados para auditoría.


In [6]:
def xml_text(
    payload
):

    try:

        root = ET.fromstring(
            payload
        )

    except Exception:

        return ""

    values = []

    for element in root.iter():

        if (
            element.text
            and
            element.text.strip()
        ):

            # Priorizar nodos textuales de Word/PowerPoint,
            # pero aceptar texto XML genérico como fallback.
            tag = (
                element.tag
                .split("}")[-1]
                .lower()
            )

            if tag in {
                "t",
                "text",
                "p"
            }:
                values.append(
                    element.text.strip()
                )

    return " ".join(
        values
    )


In [7]:
def extract_document_text(
    path
):

    result = {
        "text":
            "",

        "extraction_status":
            None,

        "package_type":
            None,

        "members_used":
            None,

        "error":
            None
    }

    try:

        with zipfile.ZipFile(
            path
        ) as z:

            names = z.namelist()

            lower_map = {
                name.lower():
                    name
                for name in names
            }

            # Caso Word OOXML normal.
            if (
                "word/document.xml"
                in lower_map
            ):

                member = lower_map[
                    "word/document.xml"
                ]

                text = xml_text(
                    z.read(
                        member
                    )
                )

                result[
                    "text"
                ] = text

                result[
                    "extraction_status"
                ] = "WORD_DOCUMENT_XML"

                result[
                    "package_type"
                ] = "WORD_PACKAGE"

                result[
                    "members_used"
                ] = member

                return result

            # Word no estándar: buscar document*.xml.
            word_candidates = [
                name
                for name
                in names
                if (
                    name.lower().startswith(
                        "word/"
                    )
                    and
                    "document"
                    in Path(
                        name
                    ).name.lower()
                    and
                    name.lower().endswith(
                        ".xml"
                    )
                )
            ]

            if word_candidates:

                texts = []

                for member in (
                    word_candidates
                ):

                    candidate_text = xml_text(
                        z.read(
                            member
                        )
                    )

                    if candidate_text:
                        texts.append(
                            candidate_text
                        )

                text = " ".join(
                    texts
                )

                if text.strip():

                    result[
                        "text"
                    ] = text

                    result[
                        "extraction_status"
                    ] = "WORD_ALTERNATE_XML"

                    result[
                        "package_type"
                    ] = "WORD_PACKAGE"

                    result[
                        "members_used"
                    ] = " | ".join(
                        word_candidates
                    )

                    return result

            # PowerPoint guardado con extensión incorrecta.
            slide_candidates = sorted(
                [
                    name
                    for name
                    in names
                    if (
                        name.lower().startswith(
                            "ppt/slides/"
                        )
                        and
                        name.lower().endswith(
                            ".xml"
                        )
                    )
                ]
            )

            if slide_candidates:

                texts = []

                for member in (
                    slide_candidates
                ):

                    candidate_text = xml_text(
                        z.read(
                            member
                        )
                    )

                    if candidate_text:
                        texts.append(
                            candidate_text
                        )

                text = " ".join(
                    texts
                )

                if text.strip():

                    result[
                        "text"
                    ] = text

                    result[
                        "extraction_status"
                    ] = "POWERPOINT_SLIDES_XML"

                    result[
                        "package_type"
                    ] = "POWERPOINT_PACKAGE"

                    result[
                        "members_used"
                    ] = " | ".join(
                        slide_candidates
                    )

                    return result

            # OpenDocument.
            content_candidates = [
                name
                for name
                in names
                if (
                    Path(
                        name
                    ).name.lower()
                    ==
                    "content.xml"
                )
            ]

            if content_candidates:

                member = (
                    content_candidates[0]
                )

                text = xml_text(
                    z.read(
                        member
                    )
                )

                if text.strip():

                    result[
                        "text"
                    ] = text

                    result[
                        "extraction_status"
                    ] = "OPEN_DOCUMENT_CONTENT_XML"

                    result[
                        "package_type"
                    ] = "OPEN_DOCUMENT_PACKAGE"

                    result[
                        "members_used"
                    ] = member

                    return result

            # Último fallback: XML con más texto.
            xml_candidates = [
                name
                for name
                in names
                if name.lower().endswith(
                    ".xml"
                )
            ]

            best_member = None
            best_text = ""

            for member in xml_candidates:

                candidate_text = xml_text(
                    z.read(
                        member
                    )
                )

                if (
                    len(
                        candidate_text
                    )
                    >
                    len(
                        best_text
                    )
                ):
                    best_member = member
                    best_text = (
                        candidate_text
                    )

            if best_text.strip():

                result[
                    "text"
                ] = best_text

                result[
                    "extraction_status"
                ] = "GENERIC_XML_FALLBACK"

                result[
                    "package_type"
                ] = "UNKNOWN_ZIP_PACKAGE"

                result[
                    "members_used"
                ] = best_member

                return result

            result[
                "extraction_status"
            ] = "NO_TEXT_RECOVERED"

            result[
                "package_type"
            ] = "UNKNOWN_ZIP_PACKAGE"

            return result

    except Exception as e:

        result[
            "extraction_status"
        ] = "EXTRACTION_ERROR"

        result[
            "error"
        ] = str(e)

        return result


## 6. Recuperar texto de los 43 documentos

In [8]:
document_records = []

document_text_cache = {}

for path in docx_files:

    extraction = (
        extract_document_text(
            path
        )
    )

    text = extraction[
        "text"
    ]

    document_text_cache[
        str(
            path
        )
    ] = text

    document_records.append({
        "file":
            str(
                path.relative_to(
                    VT_DIR
                )
            ),

        "document_name":
            path.stem,

        "text_length":
            len(
                text
            ),

        "extraction_status":
            extraction[
                "extraction_status"
            ],

        "package_type":
            extraction[
                "package_type"
            ],

        "members_used":
            extraction[
                "members_used"
            ],

        "error":
            extraction[
                "error"
            ]
    })

document_registry = pd.DataFrame(
    document_records
)

print(
    document_registry[
        "extraction_status"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nDocumentos con texto:",
    int(
        (
            document_registry[
                "text_length"
            ]
            >
            0
        ).sum()
    ),
    "/",
    len(
        document_registry
    )
)

display(
    document_registry
)


extraction_status
WORD_DOCUMENT_XML     24
WORD_ALTERNATE_XML    19
Name: count, dtype: int64

Documentos con texto: 43 / 43


,file,document_name,text_length,extraction_status,package_type,members_used,error
0,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,Baking shop,31903,WORD_DOCUMENT_XML,WORD_PACKAGE,word/document.xml,None
1,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,E-Tutor,32841,WORD_DOCUMENT_XML,WORD_PACKAGE,word/document.xml,None
2,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Food delivery.docx,Food delivery,33224,WORD_DOCUMENT_XML,WORD_PACKAGE,word/document.xml,None
3,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Gaia.docx,Gaia,32369,WORD_DOCUMENT_XML,WORD_PACKAGE,word/document.xml,None
4,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp.docx,HospitalApp,35238,WORD_DOCUMENT_XML,WORD_PACKAGE,word/document.xml,None
5,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp2.docx,HospitalApp2,32338,WORD_DOCUMENT_XML,WORD_PACKAGE,word/document.xml,None
6,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Housing.docx,Housing,33173,WORD_DOCUMENT_XML,WORD_PACKAGE,word/document.xml,None
7,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Notification.docx,Notification,32987,WORD_DOCUMENT_XML,WORD_PACKAGE,word/document.xml,None
8,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Running.docx,Running,32509,WORD_DOCUMENT_XML,WORD_PACKAGE,word/document.xml,None
9,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Travel.docx,Travel,30645,WORD_DOCUMENT_XML,WORD_PACKAGE,word/document.xml,None


## 7. Identificar ronda y fidelidad


In [9]:
def infer_round(
    file_path
):

    match = re.search(
        r"LLM Evaluation\s*(\d+)",
        str(
            file_path
        ),
        flags=re.IGNORECASE
    )

    return (
        int(
            match.group(1)
        )
        if match
        else None
    )


def infer_fidelity(
    file_path
):

    text = str(
        file_path
    ).lower()

    if (
        "high fidelity"
        in text
    ):
        return "HIGH_FIDELITY"

    if (
        "low fidelity"
        in text
    ):
        return "LOW_FIDELITY"

    return None


document_registry[
    "evaluation_round"
] = document_registry[
    "file"
].apply(
    infer_round
)

document_registry[
    "fidelity"
] = document_registry[
    "file"
].apply(
    infer_fidelity
)


## 8. Registro de URLs de metadata

In [10]:
URL_RE = re.compile(
    r"https?://[^\s<>\[\]\(\){}\"']+",
    flags=re.IGNORECASE
)


def clean_url(
    value
):

    if value is None:
        return None

    return (
        str(
            value
        )
        .strip()
        .replace(
            "\u200b",
            ""
        )
        .replace(
            "\u202f",
            ""
        )
        .replace(
            "\xa0",
            ""
        )
        .rstrip(
            ".,;:!?)]}>"
        )
    )


def normalized_url(
    value
):

    if (
        value is None
        or
        pd.isna(
            value
        )
    ):
        return None

    try:

        parsed = urlparse(
            clean_url(
                value
            )
        )

        host = (
            parsed.netloc
            .lower()
            .replace(
                "www.",
                ""
            )
        )

        path = unquote(
            parsed.path
        ).rstrip(
            "/"
        ).lower()

        return (
            host
            +
            path
        )

    except Exception:

        return None


metadata_registry = (
    metadata.copy()
)

metadata_registry[
    "normalized_url"
] = metadata_registry[
    "Scource URL"
].apply(
    normalized_url
)

metadata_registry[
    "prototype_clean"
] = metadata_registry[
    "Prototype"
].astype(str).str.strip()

display(
    metadata_registry
)


,Prototype,Prototype Type,Domain,Scource URL,normalized_url,prototype_clean
0,Food Order,High Fidelity,Delivery,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App--Community-?node-id=0-1&t=h3GPGrBwJYFFbxI3-0,figma.com/design/oiozznzhizniealb5dpkmu/food-order-app--community-,Food Order
1,Gaira (Community app),High Fidelity,Community,https://www.figma.com/design/1WJTuXkM0OfRiWjZkD61n2/Gaia-Desktop-Design-(Community)?node-id=0-1&p=f&t=kawo30RBorPkO1d2-0?,figma.com/design/1wjtuxkm0ofriwjzkd61n2/gaia-desktop-design-(community),Gaira (Community app)
2,E-Tutor,High Fidelity,Education,https://www.figma.com/community/file/1271304360859402063,figma.com/community/file/1271304360859402063,E-Tutor
3,Tracking App,High Fidelity,Mobile app,https://www.behance.net/gallery/198020739/Time-Tracker-mobile-application?tracking_source=project_owner_other_projects,behance.net/gallery/198020739/time-tracker-mobile-application,Tracking App
4,Travel,High Fidelity,Travel,https://www.figma.com/community/file/1379689864793179441,figma.com/community/file/1379689864793179441,Travel
5,Running,High Fidelity,Healthcare,https://www.behance.net/gallery/158363369/UI-Projects/modules/893495869,behance.net/gallery/158363369/ui-projects/modules/893495869,Running
6,Bakery,High Fidelity,Food Delivey,https://www.figma.com/community/file/988854619809579450,figma.com/community/file/988854619809579450,Bakery
7,Hospital 1,High Fidelity,Healthcare,https://www.figma.com/community/file/1250314760992299521,figma.com/community/file/1250314760992299521,Hospital 1
8,Hospital 2,High Fidelity,Healthcare,https://www.figma.com/community/file/12725823068122925,figma.com/community/file/12725823068122925,Hospital 2
9,Housing,High Fidelity,Real Estate,https://www.figma.com/design/QLcZcyPAov1fsHDEfnX6T4/Real-Estate-Agency---Landing-Page-Design-(Community)?node-id=0-1&p=f&t=IBZxitN7XpPIezha-0,figma.com/design/qlczcypaov1fshdefnx6t4/real-estate-agency---landing-page-design-(community),Housing


## 9. Unión por URL cuando existe evidencia directa


In [11]:
metadata_url_map = defaultdict(
    list
)

for _, row in (
    metadata_registry.iterrows()
):

    key = row[
        "normalized_url"
    ]

    if not key:
        continue

    metadata_url_map[
        key
    ].append(
        row[
            "prototype_clean"
        ]
    )


def prototypes_from_document_urls(
    text
):

    urls = [
        normalized_url(
            match
        )
        for match
        in URL_RE.findall(
            text
        )
    ]

    urls = [
        value
        for value
        in urls
        if value
    ]

    matches = []

    for url in urls:

        matches.extend(
            metadata_url_map.get(
                url,
                []
            )
        )

    return (
        sorted(
            set(
                matches
            )
        ),
        sorted(
            set(
                urls
            )
        )
    )


In [12]:
url_match_records = []

for _, row in (
    document_registry.iterrows()
):

    path = (
        VT_DIR
        /
        row[
            "file"
        ]
    )

    text = document_text_cache.get(
        str(
            path
        ),
        ""
    )

    prototypes, urls = (
        prototypes_from_document_urls(
            text
        )
    )

    url_match_records.append({
        "file":
            row[
                "file"
            ],

        "url_count":
            len(
                urls
            ),

        "urls":
            " | ".join(
                urls
            ),

        "url_prototype_matches":
            " | ".join(
                prototypes
            ),

        "url_match_count":
            len(
                prototypes
            )
    })

url_matches = pd.DataFrame(
    url_match_records
)

document_registry = (
    document_registry
    .merge(
        url_matches,
        on="file",
        how="left",
        validate="one_to_one"
    )
)

print(
    document_registry[
        "url_match_count"
    ].value_counts(
        dropna=False
    )
)


url_match_count
1    25
0    18
Name: count, dtype: int64


## 10. Construir un puente entre rondas

Cuando una evaluación de ronda 2 no contiene URL, se permite reutilizar la identidad del prototipo de la ronda 1 **solo cuando el nombre del documento es suficientemente similar y la fidelidad coincide**.

Esta unión queda marcada como `CROSS_ROUND_FILENAME`, nunca como evidencia URL directa.


In [13]:
def normalize_name(
    value
):

    value = (
        str(
            value
        )
        .lower()
        .strip()
    )

    value = re.sub(
        r"[^a-z0-9]+",
        "",
        value
    )

    for token in [
        "website",
        "app",
        "community"
    ]:

        value = value.replace(
            token,
            ""
        )

    return value


round1_identity = (
    document_registry[
        (
            document_registry[
                "evaluation_round"
            ]
            ==
            1
        )
        &
        (
            document_registry[
                "url_match_count"
            ]
            ==
            1
        )
    ]
    .copy()
)

round1_identity[
    "prototype_from_url"
] = round1_identity[
    "url_prototype_matches"
]

round1_identity[
    "document_norm"
] = round1_identity[
    "document_name"
].apply(
    normalize_name
)

display(
    round1_identity[
        [
            "document_name",
            "fidelity",
            "prototype_from_url",
            "document_norm"
        ]
    ]
)


,document_name,fidelity,prototype_from_url,document_norm
0,Baking shop,HIGH_FIDELITY,Bakery,bakingshop
1,E-Tutor,HIGH_FIDELITY,E-Tutor,etutor
4,HospitalApp,HIGH_FIDELITY,Hospital 1,hospital
5,HospitalApp2,HIGH_FIDELITY,Hospital 2,hospital2
7,Notification,HIGH_FIDELITY,Tracking App,notification
8,Running,HIGH_FIDELITY,Running,running
9,Travel,HIGH_FIDELITY,Travel,travel
10,Artwork,LOW_FIDELITY,True Art,artwork
12,Blog,LOW_FIDELITY,Blog app,blog
13,E-commerce,LOW_FIDELITY,Online Shoping,ecommerce


In [14]:
def cross_round_match(
    document_name,
    fidelity
):

    target = normalize_name(
        document_name
    )

    candidates = (
        round1_identity[
            round1_identity[
                "fidelity"
            ]
            ==
            fidelity
        ]
    )

    scored = []

    for _, row in (
        candidates.iterrows()
    ):

        score = SequenceMatcher(
            None,
            target,
            row[
                "document_norm"
            ]
        ).ratio()

        scored.append(
            (
                score,
                row[
                    "prototype_from_url"
                ],
                row[
                    "document_name"
                ]
            )
        )

    if not scored:

        return (
            None,
            np.nan,
            None
        )

    scored = sorted(
        scored,
        reverse=True,
        key=lambda x:
            x[0]
    )

    best_score, prototype, source_doc = (
        scored[0]
    )

    # Umbral conservador.
    if best_score < 0.72:

        return (
            None,
            best_score,
            source_doc
        )

    # Evitar empates casi idénticos.
    if (
        len(
            scored
        )
        >
        1
        and
        abs(
            best_score
            -
            scored[1][0]
        )
        <
        0.05
    ):

        return (
            None,
            best_score,
            source_doc
        )

    return (
        prototype,
        best_score,
        source_doc
    )


In [15]:
identity_records = []

for _, row in (
    document_registry.iterrows()
):

    if (
        row[
            "url_match_count"
        ]
        ==
        1
    ):

        identity_records.append({
            "file":
                row[
                    "file"
                ],

            "prototype":
                row[
                    "url_prototype_matches"
                ],

            "identity_status":
                "DIRECT_URL",

            "identity_score":
                1.0,

            "identity_source_doc":
                None
        })

        continue

    if (
        row[
            "evaluation_round"
        ]
        ==
        2
    ):

        (
            prototype,
            score,
            source_doc
        ) = cross_round_match(
            row[
                "document_name"
            ],
            row[
                "fidelity"
            ]
        )

        if prototype:

            identity_status = (
                "CROSS_ROUND_FILENAME"
            )

        else:

            identity_status = (
                "UNRESOLVED"
            )

        identity_records.append({
            "file":
                row[
                    "file"
                ],

            "prototype":
                prototype,

            "identity_status":
                identity_status,

            "identity_score":
                score,

            "identity_source_doc":
                source_doc
        })

        continue

    identity_records.append({
        "file":
            row[
                "file"
            ],

        "prototype":
            None,

        "identity_status":
            "UNRESOLVED",

        "identity_score":
            np.nan,

        "identity_source_doc":
            None
    })

identity_registry = pd.DataFrame(
    identity_records
)

document_registry = (
    document_registry
    .merge(
        identity_registry,
        on="file",
        how="left",
        validate="one_to_one"
    )
)

print(
    document_registry[
        "identity_status"
    ].value_counts(
        dropna=False
    )
)

display(
    document_registry[
        [
            "file",
            "evaluation_round",
            "fidelity",
            "extraction_status",
            "text_length",
            "prototype",
            "identity_status",
            "identity_score",
            "identity_source_doc"
        ]
    ]
)


identity_status
DIRECT_URL              25
UNRESOLVED              12
CROSS_ROUND_FILENAME     6
Name: count, dtype: int64


,file,evaluation_round,fidelity,extraction_status,text_length,prototype,identity_status,identity_score,identity_source_doc
0,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,1,HIGH_FIDELITY,WORD_DOCUMENT_XML,31903,Bakery,DIRECT_URL,1.000000,NaN
1,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,1,HIGH_FIDELITY,WORD_DOCUMENT_XML,32841,E-Tutor,DIRECT_URL,1.000000,NaN
2,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Food delivery.docx,1,HIGH_FIDELITY,WORD_DOCUMENT_XML,33224,NaN,UNRESOLVED,NaN,NaN
3,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Gaia.docx,1,HIGH_FIDELITY,WORD_DOCUMENT_XML,32369,NaN,UNRESOLVED,NaN,NaN
4,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp.docx,1,HIGH_FIDELITY,WORD_DOCUMENT_XML,35238,Hospital 1,DIRECT_URL,1.000000,NaN
5,LLM_Evaluations/LLM Evaluation 1/High Fidelity/HospitalApp2.docx,1,HIGH_FIDELITY,WORD_DOCUMENT_XML,32338,Hospital 2,DIRECT_URL,1.000000,NaN
6,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Housing.docx,1,HIGH_FIDELITY,WORD_DOCUMENT_XML,33173,NaN,UNRESOLVED,NaN,NaN
7,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Notification.docx,1,HIGH_FIDELITY,WORD_DOCUMENT_XML,32987,Tracking App,DIRECT_URL,1.000000,NaN
8,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Running.docx,1,HIGH_FIDELITY,WORD_DOCUMENT_XML,32509,Running,DIRECT_URL,1.000000,NaN
9,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Travel.docx,1,HIGH_FIDELITY,WORD_DOCUMENT_XML,30645,Travel,DIRECT_URL,1.000000,NaN


## 11. Heurísticas objetivo


In [16]:
HEURISTICS = {
    "H1":
        "Visibility of System Status",

    "H4":
        "Consistency and Standards",

    "H5":
        "Error Prevention",

    "H6":
        "Recognition Rather Than Recall"
}

ALL_HEADINGS = [
    "Visibility of System Status",
    "Match Between the System and the Real World",
    "User Control and Freedom",
    "Consistency and Standards",
    "Error Prevention",
    "Recognition Rather Than Recall",
    "Flexibility and Efficiency of Use",
    "Aesthetic and Minimalist Design",
    "Help Users Recognize, Diagnose, and Recover from Errors",
    "Help and Documentation"
]


## 12. Contar apariciones de los encabezados

Si un documento contiene el prompt y la respuesta, normalmente el encabezado aparecerá más de una vez. La extracción final usará la **última** aparición.


In [17]:
heading_occurrence_records = []

for _, row in (
    document_registry.iterrows()
):

    path = (
        VT_DIR
        /
        row[
            "file"
        ]
    )

    text = document_text_cache.get(
        str(
            path
        ),
        ""
    )

    normalized = re.sub(
        r"\s+",
        " ",
        text
    )

    for code, heading in (
        HEURISTICS.items()
    ):

        matches = list(
            re.finditer(
                re.escape(
                    heading
                ),
                normalized,
                flags=re.IGNORECASE
            )
        )

        heading_occurrence_records.append({
            "file":
                row[
                    "file"
                ],

            "prototype":
                row[
                    "prototype"
                ],

            "heuristic":
                code,

            "occurrences":
                len(
                    matches
                )
        })

heading_occurrences = pd.DataFrame(
    heading_occurrence_records
)

print(
    heading_occurrences[
        "occurrences"
    ].value_counts(
        dropna=False
    )
)

display(
    heading_occurrences
)


occurrences
3    65
4    45
2    24
5    17
1    12
6     7
7     2
Name: count, dtype: int64


,file,prototype,heuristic,occurrences
0,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,Bakery,H1,4
1,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,Bakery,H4,4
2,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,Bakery,H5,5
3,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Baking shop.docx,Bakery,H6,4
4,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,E-Tutor,H1,4
5,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,E-Tutor,H4,4
6,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,E-Tutor,H5,4
7,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,E-Tutor,H6,3
8,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Food delivery.docx,NaN,H1,3
9,LLM_Evaluations/LLM Evaluation 1/High Fidelity/Food delivery.docx,NaN,H4,3


## 13. Extraer la última sección de cada heurística

In [18]:
def normalize_spaces(
    text
):

    return re.sub(
        r"\s+",
        " ",
        str(
            text
        )
    ).strip()


def find_heading_positions(
    text,
    heading
):

    return [
        match.start()
        for match
        in re.finditer(
            re.escape(
                heading
            ),
            text,
            flags=re.IGNORECASE
        )
    ]


def extract_last_section(
    raw_text,
    heading
):

    text = normalize_spaces(
        raw_text
    )

    starts = find_heading_positions(
        text,
        heading
    )

    if not starts:

        return None

    start = starts[
        -1
    ]

    # Buscar el siguiente encabezado Nielsen después del
    # último encabezado objetivo.
    candidate_ends = []

    for other in ALL_HEADINGS:

        if (
            other.lower()
            ==
            heading.lower()
        ):
            continue

        for position in (
            find_heading_positions(
                text,
                other
            )
        ):

            if position > start:
                candidate_ends.append(
                    position
                )

    end = (
        min(
            candidate_ends
        )
        if candidate_ends
        else len(
            text
        )
    )

    return text[
        start:end
    ].strip()


## 14. Patrones de severidad más flexibles

In [19]:
SEVERITY_PATTERNS = [
    re.compile(
        r"Severity\s*(?:Rating)?\s*[:=\-]?\s*([0-4])(?:\.0)?\s*(?:/\s*4)?\b",
        flags=re.IGNORECASE
    ),

    re.compile(
        r"Severity\s*(?:Rating)?\s*\(?\s*([0-4])\s*\)?",
        flags=re.IGNORECASE
    ),

    re.compile(
        r"Rating\s*[:=\-]\s*([0-4])(?:\.0)?\s*(?:/\s*4)?\b",
        flags=re.IGNORECASE
    )
]

NO_ISSUE_PATTERNS = [
    re.compile(
        r"no\s+(?:major\s+)?(?:usability\s+)?issues?\s+(?:were\s+|are\s+)?(?:found|identified|detected)",
        flags=re.IGNORECASE
    ),

    re.compile(
        r"no\s+violations?\s+(?:were\s+|are\s+)?(?:found|identified|detected)",
        flags=re.IGNORECASE
    ),

    re.compile(
        r"no\s+(?:usability\s+)?problem",
        flags=re.IGNORECASE
    ),

    re.compile(
        r"issues?\s+found\s*[:\-]\s*(?:none|no issues?)",
        flags=re.IGNORECASE
    )
]


def severity_values(
    text
):

    values = []

    for pattern in (
        SEVERITY_PATTERNS
    ):

        for match in (
            pattern.finditer(
                text
            )
        ):

            try:

                value = int(
                    match.group(1)
                )

                if (
                    0
                    <=
                    value
                    <=
                    4
                ):
                    values.append(
                        value
                    )

            except Exception:
                pass

    return values


def has_explicit_no_issue(
    text
):

    return any(
        pattern.search(
            text
        )
        is not None
        for pattern
        in NO_ISSUE_PATTERNS
    )


## 15. Evidencia textual de “Issues Found”

Solo se usa como evidencia auxiliar. No convierte automáticamente cualquier texto en violación.


In [20]:
def issue_section_text(
    section
):

    match = re.search(
        r"Issues?\s+Found",
        section,
        flags=re.IGNORECASE
    )

    if not match:

        return None

    tail = section[
        match.end():
    ]

    end_match = re.search(
        r"Recommendations?",
        tail,
        flags=re.IGNORECASE
    )

    if end_match:

        tail = tail[
            :end_match.start()
        ]

    return tail.strip()


def concrete_issue_signal(
    section
):

    issue_text = issue_section_text(
        section
    )

    if not issue_text:

        return False

    lowered = issue_text.lower()

    # Descartar textos de plantilla.
    placeholder_tokens = [
        "[issue description]",
        "[file path",
        "severity rating 0-4",
        "severity rating (0-4)"
    ]

    cleaned = lowered

    for token in (
        placeholder_tokens
    ):
        cleaned = cleaned.replace(
            token,
            " "
        )

    cleaned = re.sub(
        r"[^a-z0-9]+",
        " ",
        cleaned
    ).strip()

    return (
        len(
            cleaned
        )
        >=
        25
    )


## 16. Construir candidatos por documento y heurística

Reglas conservadoras:

- `1`: existe severidad explícita mayor que 0;
- `0`: existe declaración explícita de ausencia de problemas;
- `1_WEAK`: existe una sección concreta de issues, pero sin severidad parseable;
- `NaN`: evidencia insuficiente.

`1_WEAK` se conserva para revisión y **no entra automáticamente al consenso principal**.


In [21]:
evidence_records = []

for _, row in (
    document_registry.iterrows()
):

    path = (
        VT_DIR
        /
        row[
            "file"
        ]
    )

    raw_text = document_text_cache.get(
        str(
            path
        ),
        ""
    )

    for code, heading in (
        HEURISTICS.items()
    ):

        section = extract_last_section(
            raw_text,
            heading
        )

        if not section:

            evidence_records.append({
                "file":
                    row[
                        "file"
                    ],

                "prototype":
                    row[
                        "prototype"
                    ],

                "identity_status":
                    row[
                        "identity_status"
                    ],

                "evaluation_round":
                    row[
                        "evaluation_round"
                    ],

                "heuristic":
                    code,

                "section_found":
                    False,

                "section_length":
                    0,

                "severity_values":
                    None,

                "explicit_no_issue":
                    False,

                "concrete_issue_signal":
                    False,

                "label_candidate":
                    np.nan,

                "candidate_quality":
                    "NO_SECTION",

                "section_preview":
                    None
            })

            continue

        values = severity_values(
            section
        )

        positive_values = [
            value
            for value
            in values
            if value > 0
        ]

        no_issue = has_explicit_no_issue(
            section
        )

        concrete_issue = (
            concrete_issue_signal(
                section
            )
        )

        if positive_values:

            label = 1

            quality = (
                "DIRECT_SEVERITY"
            )

        elif no_issue:

            label = 0

            quality = (
                "DIRECT_NO_ISSUE"
            )

        elif concrete_issue:

            label = np.nan

            quality = (
                "WEAK_ISSUE_TEXT"
            )

        else:

            label = np.nan

            quality = (
                "AMBIGUOUS"
            )

        evidence_records.append({
            "file":
                row[
                    "file"
                ],

            "prototype":
                row[
                    "prototype"
                ],

            "identity_status":
                row[
                    "identity_status"
                ],

            "evaluation_round":
                row[
                    "evaluation_round"
                ],

            "heuristic":
                code,

            "section_found":
                True,

            "section_length":
                len(
                    section
                ),

            "severity_values":
                str(
                    values
                ),

            "explicit_no_issue":
                no_issue,

            "concrete_issue_signal":
                concrete_issue,

            "label_candidate":
                label,

            "candidate_quality":
                quality,

            "section_preview":
                section[
                    :1000
                ]
        })

heuristic_evidence = pd.DataFrame(
    evidence_records
)

print(
    "Evidence:",
    heuristic_evidence.shape
)

print(
    "\nSection found:"
)

print(
    heuristic_evidence[
        "section_found"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nCandidate quality:"
)

print(
    heuristic_evidence[
        "candidate_quality"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nBinary candidate:"
)

print(
    heuristic_evidence[
        "label_candidate"
    ].value_counts(
        dropna=False
    )
)


Evidence: (172, 13)

Section found:
section_found
True    172
Name: count, dtype: int64

Candidate quality:
candidate_quality
DIRECT_SEVERITY    89
AMBIGUOUS          83
Name: count, dtype: int64

Binary candidate:
label_candidate
1.0    89
NaN    83
Name: count, dtype: int64


## 17. Inspección de los tres prototipos educativos

In [22]:
education_names = set(
    metadata_registry.loc[
        metadata_registry[
            "Domain"
        ]
        .astype(str)
        .str.contains(
            "education",
            case=False,
            na=False
        ),
        "prototype_clean"
    ]
)

education_evidence = (
    heuristic_evidence[
        heuristic_evidence[
            "prototype"
        ].isin(
            education_names
        )
    ]
    .copy()
)

print(
    "Educativos:",
    sorted(
        education_names
    )
)

display(
    education_evidence[
        [
            "file",
            "prototype",
            "evaluation_round",
            "identity_status",
            "heuristic",
            "severity_values",
            "explicit_no_issue",
            "concrete_issue_signal",
            "label_candidate",
            "candidate_quality",
            "section_preview"
        ]
    ]
)


Educativos: ['E-Tutor', 'Quizy', 'True Art']


,file,prototype,evaluation_round,identity_status,heuristic,severity_values,explicit_no_issue,concrete_issue_signal,label_candidate,candidate_quality,section_preview
4,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,E-Tutor,1,DIRECT_URL,H1,[],False,False,NaN,AMBIGUOUS,"Visibility of System Status: Implement loading indicators, feedback for actions, and system state visibility. Improve Match Between System and Real World: Use consistent, real-world terminology and intuitive icons. Strengthen"
5,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,E-Tutor,1,DIRECT_URL,H4,[],False,False,NaN,AMBIGUOUS,"Consistency and Standards: Standardize terminology, UI components, and pricing formats across all pages. Prevent Errors Through Better Design: Use confirmation dialogs for critical actions, restrict invalid inputs, and provide clear feedback. Reduce User Memory Load: Make filters persistent, show course details upfront, and add a ""Recently Viewed"" section. Increase"
6,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,E-Tutor,1,DIRECT_URL,H5,[],False,False,NaN,AMBIGUOUS,"error prevention, flexibility, and accessibility of support. Addressing these key issues will enhance usability and overall user experience . Note: Severity Ratings Scale 0 = Not a usability problem 1 = Cosmetic problem only 2 = Minor usability problem 3 = Major usability problem 4 = Usability catastrophe"
7,LLM_Evaluations/LLM Evaluation 1/High Fidelity/E-Tutor.docx,E-Tutor,1,DIRECT_URL,H6,"[0, 3, 2, 3, 3, 2, 0, 2]",False,True,1.0,DIRECT_SEVERITY,"Recognition Rather Than Recall Minimize user's memory load by making objects, actions, and options visible. Issues Found (Severity Rating 0-4): Hidden Course Details on Search Page Severity: 3 (Major) Users have to click on a course to view essential details such as instructor name, duration, and key topics. Displaying more information upfront would reduce the need for recall. Filter Options Are Not Persistent Severity: 2 (Minor) When users scroll through courses, filter selections are not a..."
40,LLM_Evaluations/LLM Evaluation 1/Low Fidelity/Artwork.docx,True Art,1,DIRECT_URL,H1,"[0, 3, 3, 2, 0, 3]",False,True,1.0,DIRECT_SEVERITY,"visibility of system status to enhance user experience and interaction confidence. 2. Match Between System and Real World The system should speak the users' language, with words, phrases, and concepts familiar to the user rather than system-oriented terms. Issues Found (Severity Rating 0-4): Unclear Labels and Terminology: Severity: 3 (Major) Terms like “Title” and “Label” are generic and not meaningful to users. They don’t relate to familiar artwork or gallery terms. Icons Not Following Rea..."
41,LLM_Evaluations/LLM Evaluation 1/Low Fidelity/Artwork.docx,True Art,1,DIRECT_URL,H4,"[0, 3, 3, 2, 0, 2]",False,True,1.0,DIRECT_SEVERITY,"Consistency and Standards Users should not have to wonder whether different words, situations, or actions mean the same thing. Issues Found (Severity Rating 0-4): Inconsistent Terminology: Severity: 3 (Major) Labels such as “Title,” “Label,” and “Description” are used inconsistently across different pages, leading to confusion. Inconsistent UI Patterns: Severity: 3 (Major) Navigation elements (e.g., bottom nav bar and sort options) appear in varying positions and styles across screens, break..."
42,LLM_Evaluations/LLM Evaluation 1/Low Fidelity/Artwork.docx,True Art,1,DIRECT_URL,H5,[],False,False,NaN,AMBIGUOUS,error prevention and user safety. 6.
43,LLM_Evaluations/LLM Evaluation 1/Low Fidelity/Artwork.docx,True Art,1,DIRECT_URL,H6,"[0, 3, 3, 2, 0, 2]",False,True,1.0,DIRECT_SEVERITY,"Recognition Rather Than Recall Minimize user's memory load by making objects, actions, and options visible. Issues Found (Severity Rating 0-4): Hidden Options and Actions: Severity: 3 (Major) Sorting and filtering options are not immediately visible and require users to recall where they are located. Lack of Instructions or Guidance: Severity: 3 (Major) No t

## 18. Consenso únicamente con evidencia directa

Solo entran:

- identidad `DIRECT_URL` o `CROSS_ROUND_FILENAME`;
- `label_candidate` 0/1;
- calidad `DIRECT_SEVERITY` o `DIRECT_NO_ISSUE`.

Los textos débiles no se convierten en etiquetas.


In [23]:
valid_direct = (
    heuristic_evidence[
        heuristic_evidence[
            "prototype"
        ].notna()
        &
        heuristic_evidence[
            "label_candidate"
        ].notna()
        &
        heuristic_evidence[
            "candidate_quality"
        ].isin(
            [
                "DIRECT_SEVERITY",
                "DIRECT_NO_ISSUE"
            ]
        )
    ]
    .copy()
)

valid_direct[
    "label_candidate"
] = valid_direct[
    "label_candidate"
].astype(
    int
)

print(
    "Evidencia directa válida:",
    len(
        valid_direct
    )
)

print(
    "Prototipos:",
    valid_direct[
        "prototype"
    ].nunique()
)

print(
    valid_direct[
        "label_candidate"
    ].value_counts(
        dropna=False
    )
)


Evidencia directa válida: 63
Prototipos: 15
label_candidate
1    63
Name: count, dtype: int64


In [24]:
def consensus_for_group(
    group
):

    labels = (
        group[
            "label_candidate"
        ]
        .astype(int)
        .tolist()
    )

    n = len(
        labels
    )

    positive = sum(
        value == 1
        for value
        in labels
    )

    negative = sum(
        value == 0
        for value
        in labels
    )

    if positive > negative:

        consensus = 1
        status = "MAJORITY_POSITIVE"

    elif negative > positive:

        consensus = 0
        status = "MAJORITY_NEGATIVE"

    else:

        consensus = np.nan
        status = "TIE"

    return pd.Series({
        "n_evaluations":
            n,

        "n_positive":
            positive,

        "n_negative":
            negative,

        "agreement_ratio":
            (
                max(
                    positive,
                    negative
                )
                /
                n
                if n
                else np.nan
            ),

        "consensus_candidate":
            consensus,

        "consensus_status":
            status
    })


if valid_direct.empty:

    consensus_long = pd.DataFrame(
        columns=[
            "prototype",
            "heuristic",
            "n_evaluations",
            "n_positive",
            "n_negative",
            "agreement_ratio",
            "consensus_candidate",
            "consensus_status"
        ]
    )

else:

    rows = []

    for (
        prototype,
        heuristic
    ), group in valid_direct.groupby(
        [
            "prototype",
            "heuristic"
        ]
    ):

        result = (
            consensus_for_group(
                group
            )
        )

        row = {
            "prototype":
                prototype,

            "heuristic":
                heuristic
        }

        row.update(
            result.to_dict()
        )

        rows.append(
            row
        )

    consensus_long = pd.DataFrame(
        rows
    )

display(
    consensus_long
)


,prototype,heuristic,n_evaluations,n_positive,n_negative,agreement_ratio,consensus_candidate,consensus_status
0,Blog app,H4,1,1,0,1.0,1,MAJORITY_POSITIVE
1,Blog app,H6,1,1,0,1.0,1,MAJORITY_POSITIVE
2,E-Tutor,H6,1,1,0,1.0,1,MAJORITY_POSITIVE
3,Fintech Mobile App,H5,1,1,0,1.0,1,MAJORITY_POSITIVE
4,Fintech Mobile App,H6,2,2,0,1.0,1,MAJORITY_POSITIVE
5,Food Delivery,H5,1,1,0,1.0,1,MAJORITY_POSITIVE
6,Food Delivery,H6,2,2,0,1.0,1,MAJORITY_POSITIVE
7,Healthcare App,H1,1,1,0,1.0,1,MAJORITY_POSITIVE
8,Healthcare App,H4,1,1,0,1.0,1,MAJORITY_POSITIVE
9,Healthcare App,H6,2,2,0,1.0,1,MAJORITY_POSITIVE


In [25]:
if consensus_long.empty:

    consensus_wide = pd.DataFrame(
        columns=[
            "prototype",
            "H1_candidate",
            "H4_candidate",
            "H5_candidate",
            "H6_candidate"
        ]
    )

else:

    consensus_wide = (
        consensus_long
        .pivot(
            index="prototype",
            columns="heuristic",
            values="consensus_candidate"
        )
        .reset_index()
    )

    consensus_wide.columns.name = None

    consensus_wide = (
        consensus_wide.rename(
            columns={
                "H1":
                    "H1_candidate",

                "H4":
                    "H4_candidate",

                "H5":
                    "H5_candidate",

                "H6":
                    "H6_candidate"
            }
        )
    )

display(
    consensus_wide
)


,prototype,H1_candidate,H4_candidate,H5_candidate,H6_candidate
0,Blog app,NaN,1.0,NaN,1.0
1,E-Tutor,NaN,NaN,NaN,1.0
2,Fintech Mobile App,NaN,NaN,1.0,1.0
3,Food Delivery,NaN,NaN,1.0,1.0
4,Healthcare App,1.0,1.0,NaN,1.0
5,Hospital 1,NaN,1.0,NaN,1.0
6,Hospital 2,1.0,1.0,NaN,1.0
7,KwikTan Membership,1.0,1.0,NaN,1.0
8,Online Shoping,1.0,1.0,1.0,1.0
9,Quizy,1.0,1.0,NaN,1.0


## 19. Decisión de usabilidad de la fuente

In [26]:
n_direct_labels = len(
    valid_direct
)

n_consensus_prototypes = (
    consensus_wide[
        "prototype"
    ].nunique()
    if (
        not consensus_wide.empty
        and
        "prototype"
        in consensus_wide.columns
    )
    else 0
)

source_decision = pd.DataFrame([
    {
        "dataset":
            "29042675",

        "irr_compare_as_target":
            False,

        "llm_documents":
            len(
                document_registry
            ),

        "documents_with_text":
            int(
                (
                    document_registry[
                        "text_length"
                    ]
                    >
                    0
                ).sum()
            ),

        "identified_documents":
            int(
                document_registry[
                    "prototype"
                ].notna().sum()
            ),

        "direct_binary_label_evidence":
            n_direct_labels,

        "consensus_prototypes":
            n_consensus_prototypes,

        "label_role":
            (
                "LLM_DIRECT_CANDIDATE"
                if n_direct_labels > 0
                else "LABEL_REFERENCE"
            ),

        "canonical_web_X_ready":
            False,

        "primary_training_ready":
            False,

        "reason":
            "IRR Compare representa acuerdo y no Y de violación. "
            "Las evaluaciones LLM pueden aportar Y directa si existe evidencia textual parseable, "
            "pero las fuentes Figma/Behance/SharePoint siguen sin aportar X web canónica."
    }
])

display(
    source_decision
)


,dataset,irr_compare_as_target,llm_documents,documents_with_text,identified_documents,direct_binary_label_evidence,consensus_prototypes,label_role,canonical_web_X_ready,primary_training_ready,reason
0,29042675,False,43,43,31,63,15,LLM_DIRECT_CANDIDATE,False,False,"IRR Compare representa acuerdo y no Y de violación. Las evaluaciones LLM pueden aportar Y directa si existe evidencia textual parseable, pero las fuentes Figma/Behance/SharePoint siguen sin aportar X web canónica."


## 20. Guardar

In [27]:
package_audit.to_csv(
    REPORTS_DIR
    / "vt_docx_package_audit.csv",
    index=False
)

document_registry.to_csv(
    REPORTS_DIR
    / "vt_document_recovery_registry.csv",
    index=False
)

heading_occurrences.to_csv(
    REPORTS_DIR
    / "vt_heading_occurrence_audit.csv",
    index=False
)

heuristic_evidence.to_csv(
    INTERIM_DIR
    / "vt_recovered_heuristic_evidence.csv",
    index=False
)

education_evidence.to_csv(
    REPORTS_DIR
    / "vt_recovered_education_evidence.csv",
    index=False
)

valid_direct.to_csv(
    INTERIM_DIR
    / "vt_recovered_direct_label_evidence.csv",
    index=False
)

consensus_long.to_csv(
    INTERIM_DIR
    / "vt_recovered_candidate_consensus_long.csv",
    index=False
)

consensus_wide.to_csv(
    INTERIM_DIR
    / "vt_recovered_candidate_targets.csv",
    index=False
)

source_decision.to_csv(
    REPORTS_DIR
    / "vt_recovery_source_decision.csv",
    index=False
)


In [28]:
expected = [
    REPORTS_DIR
    / "vt_docx_package_audit.csv",

    REPORTS_DIR
    / "vt_document_recovery_registry.csv",

    REPORTS_DIR
    / "vt_heading_occurrence_audit.csv",

    INTERIM_DIR
    / "vt_recovered_heuristic_evidence.csv",

    REPORTS_DIR
    / "vt_recovered_education_evidence.csv",

    INTERIM_DIR
    / "vt_recovered_direct_label_evidence.csv",

    INTERIM_DIR
    / "vt_recovered_candidate_consensus_long.csv",

    INTERIM_DIR
    / "vt_recovered_candidate_targets.csv",

    REPORTS_DIR
    / "vt_recovery_source_decision.csv"
]

for path in expected:

    print(
        path.name,
        "✅"
        if path.exists()
        else "❌"
    )


vt_docx_package_audit.csv ✅
vt_document_recovery_registry.csv ✅
vt_heading_occurrence_audit.csv ✅
vt_recovered_heuristic_evidence.csv ✅
vt_recovered_education_evidence.csv ✅
vt_recovered_direct_label_evidence.csv ✅
vt_recovered_candidate_consensus_long.csv ✅
vt_recovered_candidate_targets.csv ✅
vt_recovery_source_decision.csv ✅


## Punto de detención

No renombrar todavía `H1_candidate`, `H4_candidate`, `H5_candidate` o `H6_candidate` como targets finales.

Los resultados decisivos de este notebook son:

- distribución de `extraction_status`;
- número de documentos con texto recuperado;
- distribución de `identity_status`;
- número de apariciones de cada encabezado;
- distribución de `candidate_quality`;
- número de filas en `valid_direct`;
- número de prototipos con consenso.

Después de revisar esos valores se decidirá si `29042675` puede conservarse como fuente `LLM_DIRECT` o debe quedar como `LABEL_REFERENCE`.
